# v93 tabnn (FT-Transformer) — Kaggle T4 training

Complementary **tabular NN leg** for v93 (`tabnn`). Per-row FT-Transformer regressor,
struct-free features, `GroupKFold(5)` by well (identical to L1 GBM / L2 TCN / L5), 3 seeds,
3-seed-ensemble OOF. Different model family from the GBM/TCN legs -> decorrelation.

**Self-contained** — inlines model + training + data-load + feature-selection from
`train_tabnn_v93.py` (nothing imported from the repo).

## How to run on Kaggle
1. **Upload `v93/artifacts_v93/` AS-IS as a Kaggle Dataset** (it holds
   `data/train.parquet` ~8.3GB, `features_l1.json`, `feat_rank_nn.json`, ...).
   Attach that dataset to this notebook. **No competition data is needed** — tabnn only reads the parquet + 2 json.
2. Enable the **GPU T4** accelerator.
3. Run the cells in order. **Run the BENCHMARK cell first** to see the extrapolated ETA
   (ms/step, per-epoch minutes, total hours) before committing to full training.
4. Run the TRAINING cell. It is **resumable** — a `(seed,fold)` whose checkpoint + val-pred
   already exist is skipped.
5. The **COMPLEMENTARITY cell** right after training tells you — on Kaggle, no download needed —
   whether tabnn de-correlates from the GBM / TCN / L5 legs and improves the blend.
6. Outputs land in `/kaggle/working` (see the last markdown cell for what to download).

## Default = light 1-fold complementarity check
The config cell defaults to **`SEEDS=[0]`, `FOLDS_TO_RUN=[0]`** — 1 seed x fold0 — so you can quickly see
on Kaggle whether tabnn is complementary to the other legs. For the **FULL OOF** set
`SEEDS = [0, 1, 2]` and `FOLDS_TO_RUN = list(range(5))` in the config cell.

## T4 AMP note
T4 (Turing sm_75) has **FP16** tensor cores but **not bf16**, so the default is `AMP="fp16"`
with a `GradScaler` (fp16 requires loss scaling). `AMP="bf16"` / `AMP="off"` are also supported.

In [ ]:
# =====================================================================
# Imports + CONFIG (edit the plain vars below; env vars override if set) + path detection
# =====================================================================
import os, io, sys, json, math, time, gc, copy, re, glob, contextlib, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
import pyarrow.parquet as pq
from sklearn.model_selection import GroupKFold

# ---------------- EDIT ME: config (plain Python vars; os.environ overrides) ----------------
TOPK        = int(os.environ.get("TOPK",     "256"))   # 0 = all features; attention is O(TOPK^2)
DIM         = int(os.environ.get("DIM",      "96"))    # token dim d
N_BLOCK     = int(os.environ.get("NBLOCK",   "3"))     # transformer blocks
N_HEAD      = int(os.environ.get("NHEAD",    "8"))     # attention heads (DIM % N_HEAD == 0)
FF_MULT     = int(os.environ.get("FFMULT",   "2"))     # FFN hidden = FF_MULT * d
DROP        = float(os.environ.get("DROP",   "0.1"))
BATCH       = int(os.environ.get("BATCH",    "1024"))  # T4 16GB; fp16 attn @TOPK256 -> keep modest, can raise
EPOCHS      = int(os.environ.get("EPOCHS",   "40"))
PATIENCE    = int(os.environ.get("PATIENCE", "6"))     # early stop
LR          = float(os.environ.get("LR",     "1e-3"))
WD          = float(os.environ.get("WD",     "1e-5"))
EMA_DECAY   = float(os.environ.get("EMA",    "0.999"))
HUBER_DELTA = float(os.environ.get("HUBER",  "1.0"))   # Huber delta on standardized target
CLIP        = float(os.environ.get("CLIP",   "1.0"))
WARM_FRAC   = 0.05
AMP         = os.environ.get("AMP", "fp16").lower()    # "fp16"(T4 default) | "bf16" | "off"
N_SPLITS    = 5

# ---- run scope: default = 1 seed x fold0 (light complementarity check, finishes on Kaggle) ----
# FULL run: set  SEEDS = [0, 1, 2]  and  FOLDS_TO_RUN = list(range(5))
SEEDS       = [int(s) for s in os.environ.get("SEEDS", "0").split(",") if s.strip() != ""]   # default [0]
FOLDS_TO_RUN = [int(s) for s in os.environ.get("FOLDS", "0").split(",") if s.strip() != ""]  # default [0] = fold0 only
# FOLDS_TO_RUN = list(range(N_SPLITS))   # <- uncomment for all 5 folds

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

# ---------------- AMP: context + GradScaler policy ----------------
# fp16 REQUIRES loss scaling (GradScaler); bf16 / off do NOT use a scaler.
if DEVICE != "cuda":
    AMP = "off"
_AMP_DTYPE = {"fp16": torch.float16, "bf16": torch.bfloat16}.get(AMP, None)
def amp_ctx():
    if DEVICE == "cuda" and _AMP_DTYPE is not None:
        return torch.autocast("cuda", dtype=_AMP_DTYPE)
    return contextlib.nullcontext()
USE_SCALER = (DEVICE == "cuda" and AMP == "fp16")   # only fp16 needs the GradScaler

# ---------------- path detection: uploaded artifacts_v93 as a Kaggle dataset ----------------
def _detect_art():
    hits = (glob.glob("/kaggle/input/*/features_l1.json")
            + glob.glob("/kaggle/input/*/*/features_l1.json"))
    if hits:
        return Path(hits[0]).parent                # ART = parent of the found features_l1.json
    # local fallback (also runs on the dev box)
    return Path(r"C:\Users\kosaka256\Documents\rogii_claude\v93\artifacts_v93")

ART        = _detect_art()
PARQUET    = ART / "data" / "train.parquet"
FEATS_JSON = ART / "features_l1.json"
RANK       = ART / "feat_rank_nn.json"
WORK       = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./tabnn_out")
WORK.mkdir(parents=True, exist_ok=True)
MODELS_DIR  = WORK / "tabnn_models";  MODELS_DIR.mkdir(parents=True, exist_ok=True)
VALPRED_DIR = WORK / "tabnn_valpred"; VALPRED_DIR.mkdir(parents=True, exist_ok=True)

print("ART       =", ART)
print("PARQUET   =", PARQUET, "(exists)" if PARQUET.exists() else "(MISSING)")
print("FEATS_JSON=", FEATS_JSON, "(exists)" if FEATS_JSON.exists() else "(MISSING)")
print("RANK      =", RANK, "(exists)" if RANK.exists() else "(MISSING)")
print("WORK      =", WORK)
print("device    =", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "-",
      "| torch", torch.__version__)
print("AMP       =", AMP, "| USE_SCALER (fp16 GradScaler) =", USE_SCALER)
print("cfg: TOPK=%d DIM=%d NBLOCK=%d NHEAD=%d FFMULT=%d DROP=%.2f BATCH=%d EPOCHS=%d PATIENCE=%d"
      % (TOPK, DIM, N_BLOCK, N_HEAD, FF_MULT, DROP, BATCH, EPOCHS, PATIENCE))
print("run scope: SEEDS=%s FOLDS_TO_RUN=%s  (%d seed x %d fold = %d total)  [FULL = SEEDS[0,1,2] x FOLDS range(5)]"
      % (SEEDS, FOLDS_TO_RUN, len(SEEDS), len(FOLDS_TO_RUN), len(SEEDS)*len(FOLDS_TO_RUN)))


In [ ]:
# =====================================================================
# Feature selection (struct-free, drop last_known_tvt, TOPK via feat_rank_nn.json)
# + memory-safe data load + robust scale + target standardize + GroupKFold folds
# =====================================================================
features_all = json.loads(Path(FEATS_JSON).read_text())
assert isinstance(features_all, list), "features_l1.json must be a list of column names"
# struct-free guard (no gf_/v50b/struct/v38/v41 ever)
_forbidden = [c for c in features_all if re.search(r"(^gf_|v50b|struct|v38|v41)", c)]
assert not _forbidden, f"struct/gf/v50b leaked into features_l1.json: {_forbidden[:10]}"

feats_base = [c for c in features_all if c != "last_known_tvt"]   # drop leaky last_known_tvt
if TOPK and TOPK > 0:
    rank = json.loads(Path(RANK).read_text())
    assert isinstance(rank, list), "feat_rank_nn.json must be a ranked list of column names"
    rank = [c for c in rank if c != "last_known_tvt" and c in set(feats_base)]
    feats_nn = rank[:TOPK]
    if len(feats_nn) < TOPK:                       # pad if rank shorter than TOPK (normally never)
        for c in feats_base:
            if c not in feats_nn:
                feats_nn.append(c)
            if len(feats_nn) >= TOPK:
                break
    print(f"TOPK={TOPK}: selected top-{len(feats_nn)} cols from feat_rank_nn.json")
else:
    feats_nn = feats_base
    print(f"all features: {len(feats_nn)} cols (last_known_tvt dropped)")
# re-assert struct-free after selection
assert not [c for c in feats_nn if re.search(r"(^gf_|v50b|struct|v38|v41)", c)], "forbidden col after selection"

# ---- memory-safe data load: preallocate X, read parquet in column chunks ----
_pq_names = set(pq.read_schema(PARQUET).names)
assert all(c in _pq_names for c in feats_nn), "all selected features must exist in parquet"
META = ["well", "target"]

def _load_full():
    N = pq.read_metadata(PARQUET).num_rows
    meta_tbl = pq.read_table(PARQUET, columns=META)
    g = meta_tbl.column("well").to_pandas().to_numpy()
    y = meta_tbl.column("target").to_numpy(zero_copy_only=False).astype(np.float64)
    del meta_tbl; gc.collect()
    X = np.empty((N, len(feats_nn)), np.float32)   # TOPK=256 -> ~3.9GB
    CH = 64
    for s in range(0, len(feats_nn), CH):
        cols = feats_nn[s:s + CH]
        t = pq.read_table(PARQUET, columns=cols)
        for jj in range(len(cols)):
            X[:, s + jj] = t.column(jj).to_numpy(zero_copy_only=False).astype(np.float32)
        del t; gc.collect()
    return N, g, y, X

t0 = time.time()
N, g, y, X = _load_full()
F_IN = X.shape[1]
print(f"loaded rows={N} wells={len(pd.unique(g))} F={F_IN} ({time.time()-t0:.0f}s)")

# ---- folds: identical construction to L1 GBM / L2 TCN / L5 ----
folds = list(GroupKFold(n_splits=N_SPLITS).split(np.zeros(N), y, groups=g))

# ---- robust standardize: stats on a row subsample, transform in-place (avoid 11GB copies) ----
samp = np.arange(0, N, max(1, N // 200000))
Xs = X[samp]
med = np.nanmedian(Xs, axis=0).astype(np.float32)
q75 = np.nanpercentile(Xs, 75, axis=0).astype(np.float32)
q25 = np.nanpercentile(Xs, 25, axis=0).astype(np.float32)
scl = np.maximum((q75 - q25) / 1.349, 1e-6).astype(np.float32)
del Xs; gc.collect()
X -= med                                        # in-place broadcast
X /= scl                                        # in-place
np.nan_to_num(X, copy=False, nan=0.0, posinf=8.0, neginf=-8.0)
np.clip(X, -8.0, 8.0, out=X)                    # in-place

# ---- target standardize (decode preds back to target units) ----
t_mean = float(y.mean()); t_scale = float(y.std())
yn = ((y - t_mean) / t_scale).astype(np.float32)
print(f"rows {N} | F {F_IN} | t_mean/scale {round(t_mean,3)}/{round(t_scale,3)}")


In [ ]:
# =====================================================================
# FT-Transformer: per-feature tokenizer + [CLS] + N pre-LN transformer blocks (SDPA) + MLP head
# + EMA + cosine LR + batched predict_rows (uses amp_ctx). GradScaler policy set in the config cell.
# =====================================================================
class FeatureTokenizer(nn.Module):
    """Each continuous feature x_j -> token = x_j * W_j + b_j ; W,b are (F,d) learned params."""
    def __init__(self, n_feat, d):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_feat, d))
        self.bias   = nn.Parameter(torch.empty(n_feat, d))
        nn.init.normal_(self.weight, std=0.02)
        nn.init.normal_(self.bias, std=0.02)
    def forward(self, x):                            # x: (B, F)
        return x.unsqueeze(-1) * self.weight + self.bias   # (B, F, d)


class TransformerBlock(nn.Module):
    """pre-LN: MHSA (SDPA) + GELU-FFN, residual + dropout."""
    def __init__(self, d, n_head, ff_mult, drop):
        super().__init__()
        assert d % n_head == 0, "DIM must be divisible by N_HEAD"
        self.h = n_head; self.dk = d // n_head; self.p = drop
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.ln2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, ff_mult * d), nn.GELU(),
                                nn.Dropout(drop), nn.Linear(ff_mult * d, d))
        self.do = nn.Dropout(drop)
    def forward(self, x):                            # x: (B, T, d)
        B, T, D = x.shape
        h = self.ln1(x)
        q, k, v = self.qkv(h).chunk(3, dim=-1)
        q = q.view(B, T, self.h, self.dk).transpose(1, 2)
        k = k.view(B, T, self.h, self.dk).transpose(1, 2)
        v = v.view(B, T, self.h, self.dk).transpose(1, 2)
        a = F.scaled_dot_product_attention(q, k, v, dropout_p=self.p if self.training else 0.0)
        a = self.proj(a.transpose(1, 2).reshape(B, T, D))
        x = x + self.do(a)
        x = x + self.do(self.ff(self.ln2(x)))
        return x


class FTTransformer(nn.Module):
    def __init__(self, n_feat, d, n_block, n_head, ff_mult, drop):
        super().__init__()
        self.tok = FeatureTokenizer(n_feat, d)
        self.cls = nn.Parameter(torch.empty(1, 1, d)); nn.init.normal_(self.cls, std=0.02)
        self.blocks = nn.ModuleList([TransformerBlock(d, n_head, ff_mult, drop)
                                     for _ in range(n_block)])
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(),
                                  nn.Dropout(drop), nn.Linear(d, 1))
    def forward(self, x):                            # x: (B, F)
        t = self.tok(x)                              # (B, F, d)
        cls = self.cls.expand(t.shape[0], -1, -1)
        t = torch.cat([cls, t], dim=1)               # (B, F+1, d)
        for blk in self.blocks:
            t = blk(t)
        return self.head(t[:, 0]).squeeze(-1)        # CLS -> (B,)


class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
    def state_dict(self):
        return {k: v.clone() for k, v in self.shadow.items()}


def new_model():
    return FTTransformer(F_IN, DIM, N_BLOCK, N_HEAD, FF_MULT, DROP).to(DEVICE)


def cosine_lr(step, total, warm):
    if step < warm:
        return step / max(1, warm)
    t = (step - warm) / max(1, total - warm)
    return 0.5 * (1.0 + math.cos(math.pi * t))


@torch.no_grad()
def predict_rows(model, idx):
    """idx (row-index ndarray) -> preds in STANDARDIZED-target units (ndarray). Batched + amp."""
    model.eval()
    out = np.empty(len(idx), np.float32)
    for i in range(0, len(idx), BATCH):
        bidx = idx[i:i + BATCH]
        xb = torch.from_numpy(X[bidx]).to(DEVICE)
        with amp_ctx():
            pb = model(xb)
        out[i:i + len(bidx)] = pb.float().cpu().numpy()
    return out


_tmp = new_model()
N_PARAM = sum(p.numel() for p in _tmp.parameters())
print(f"model: FT-Transformer d={DIM} blocks={N_BLOCK} heads={N_HEAD} ff={FF_MULT}*d "
      f"tokens={F_IN + 1}(CLS+{F_IN}) params={N_PARAM:,}")
del _tmp; gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# =====================================================================
# SPEED BENCHMARK  (run this FIRST to gauge feasibility before committing)
# proper warmup (>=15 steps) + cudnn.benchmark + cuda.synchronize() around timing.
# Uses the SAME AMP/GradScaler path as real training so the ETA is realistic.
# =====================================================================
assert "new_model" in globals() and "folds" in globals(), "run the model + data cells first"
_bm  = new_model()
_opt = torch.optim.AdamW(_bm.parameters(), lr=LR, weight_decay=WD)
_sc  = torch.cuda.amp.GradScaler(enabled=USE_SCALER)
_xb  = torch.randn(BATCH, F_IN, device=DEVICE)
_yb  = torch.randn(BATCH, device=DEVICE)

def _train_step():
    _opt.zero_grad(set_to_none=True)
    with amp_ctx():
        p = _bm(_xb)
    l = F.huber_loss(p.float(), _yb, delta=HUBER_DELTA)
    if USE_SCALER:                       # fp16 path: scale -> unscale -> clip -> step -> update
        _sc.scale(l).backward()
        _sc.unscale_(_opt)
        torch.nn.utils.clip_grad_norm_(_bm.parameters(), CLIP)
        _sc.step(_opt); _sc.update()
    else:                                # bf16 / off: no scaler
        l.backward()
        torch.nn.utils.clip_grad_norm_(_bm.parameters(), CLIP)
        _opt.step()

_WARM, _NB = 20, 50
_bm.train()
for _ in range(_WARM):                   # warmup (>=15)
    _train_step()
if DEVICE == "cuda": torch.cuda.synchronize()
_t = time.time()
for _ in range(_NB):
    _train_step()
if DEVICE == "cuda": torch.cuda.synchronize()
tr_ms = (time.time() - _t) / _NB * 1000

_bm.eval()
with torch.no_grad():
    for _ in range(_WARM):
        with amp_ctx(): _ = _bm(_xb)
    if DEVICE == "cuda": torch.cuda.synchronize()
    _t = time.time()
    for _ in range(_NB):
        with amp_ctx(): _ = _bm(_xb)
    if DEVICE == "cuda": torch.cuda.synchronize()
fw_ms = (time.time() - _t) / _NB * 1000

_k0 = FOLDS_TO_RUN[0]
n_tr = len(folds[_k0][0]); n_va = len(folds[_k0][1])
sfit = math.ceil(n_tr / BATCH); sval = math.ceil(n_va / BATCH)
ep_s = (sfit * tr_ms + sval * fw_ms) / 1000.0
nfold_run  = len(SEEDS) * len(FOLDS_TO_RUN)   # current scope (default 1)
nfold_full = 3 * N_SPLITS                      # full 3seed x 5fold = 15

print(f"GPU {torch.cuda.get_device_name(0) if DEVICE=='cuda' else DEVICE} | AMP={AMP} "
      f"USE_SCALER={USE_SCALER} | BATCH={BATCH} F_IN={F_IN}")
print(f"train {tr_ms:.1f} ms/step | fwd {fw_ms:.1f} ms/step | steps/epoch: fit={sfit} val={sval}")
print(f"per-epoch ~ {ep_s:.1f}s ({ep_s/60:.2f} min)")
print(f"current scope = {len(SEEDS)} seed x {len(FOLDS_TO_RUN)} fold = {nfold_run} folds | FULL = {nfold_full} folds")
print("ETA table (assumes every fold runs the full epoch budget; early-stop makes it faster):")
for E in (15, 25, 40):
    print(f"  {E:>2d} epochs -> {ep_s*E/60:6.1f} min/fold | this scope ~{ep_s*E*nfold_run/3600:5.2f} h "
          f"({nfold_run} folds) | FULL ~{ep_s*E*nfold_full/3600:5.2f} h ({nfold_full} folds)")

del _bm, _opt, _sc, _xb, _yb; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()


In [ ]:
# =====================================================================
# TRAINING: seed x fold loop. early stop + EMA. per-epoch progress + running ETA.
# Saves EACH (seed,fold) model state_dict to WORK/tabnn_models/tab_s{seed}_f{fold}.pt.
# Resumable: skips a (seed,fold) whose model .pt AND val-pred cache both exist.
# =====================================================================
folds_to_run = FOLDS_TO_RUN            # default [0] (1-fold); FULL = list(range(5))
seed_oofs = {}
_run_t0 = time.time()
_folds_total = len(SEEDS) * len(folds_to_run)
_folds_done = 0
print(f"[tabnn] AMP={AMP} USE_SCALER={USE_SCALER} BATCH={BATCH} EPOCHS={EPOCHS} "
      f"total {_folds_total} folds (={len(SEEDS)} seed x {len(folds_to_run)} fold)", flush=True)


def train_fold(seed, k):
    """Train one (seed,fold). Returns (best_valpred_TARGET, best_rmse, va_idx, best_ema_state_dict)."""
    tr, va = folds[k]
    tr = np.asarray(tr); va = np.asarray(va)
    torch.manual_seed(1000 * seed + k)
    rng = np.random.default_rng(1000 * seed + k)

    model = new_model()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    ema = EMA(model, EMA_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_SCALER)   # fp16 loss scaling (no-op otherwise)
    steps_ep = math.ceil(len(tr) / BATCH)
    total = steps_ep * EPOCHS; warm = int(total * WARM_FRAC); step = 0
    best = (1e18, None); best_sd = None; bad = 0

    for ep in range(EPOCHS):
        ep_t0 = time.time()
        model.train()
        perm = rng.permutation(len(tr))
        for i in range(0, len(perm), BATCH):
            bidx = tr[perm[i:i + BATCH]]
            xb = torch.from_numpy(X[bidx]).to(DEVICE)
            yb = torch.from_numpy(yn[bidx]).to(DEVICE)
            for pg in opt.param_groups:
                pg["lr"] = LR * cosine_lr(step, total, warm)
            opt.zero_grad(set_to_none=True)
            with amp_ctx():
                pred = model(xb)
            loss = F.huber_loss(pred.float(), yb, delta=HUBER_DELTA)
            if USE_SCALER:                      # fp16: scale -> unscale -> clip -> step -> update
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                scaler.step(opt)
                scaler.update()
            else:                               # bf16 / off: plain backward, no scaler
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                opt.step()
            ema.update(model); step += 1
        if not torch.isfinite(loss).item():
            raise RuntimeError(f"loss non-finite (seed{seed} fold{k} ep{ep})")

        # ---- evaluate with EMA weights (restore raw weights afterwards) ----
        bak = copy.deepcopy(model.state_dict())
        sd = model.state_dict()
        for kk, vv in ema.state_dict().items():
            sd[kk].copy_(vv)
        vp = predict_rows(model, va) * t_scale + t_mean
        rm = float(np.sqrt(np.mean((vp - y[va]) ** 2)))
        if not np.isfinite(rm):
            raise RuntimeError(f"val RMSE non-finite (seed{seed} fold{k} ep{ep})")
        if rm < best[0] - 1e-5:
            best = (rm, vp.copy()); bad = 0
            best_sd = {kk: vv.detach().cpu().clone() for kk, vv in ema.state_dict().items()}
        else:
            bad += 1
        model.load_state_dict(bak)
        _el = time.time() - _run_t0
        print(f"  seed{seed} fold{k} ep{ep+1}/{EPOCHS} val={rm:.4f} best={best[0]:.4f} "
              f"bad={bad}/{PATIENCE} ({time.time()-ep_t0:.0f}s/ep) [elapsed {_el/60:.1f}min]", flush=True)
        if bad >= PATIENCE:
            break

    del model, opt, ema; gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return best[1], best[0], va, best_sd


for seed in SEEDS:
    oof = np.zeros(N, np.float64); filled = np.zeros(N, bool); tt = time.time()
    for k in folds_to_run:
        cache = VALPRED_DIR / f"tab_s{seed}_f{k}_valpred.npz"
        mpath = MODELS_DIR  / f"tab_s{seed}_f{k}.pt"
        if cache.exists() and mpath.exists():                 # resume: skip completed (seed,fold)
            z = np.load(cache)
            oof[z["rows"]] = z["pred"]; filled[z["rows"]] = True
            print(f"seed{seed} fold{k} [resume: model+cache exist] val={len(z['rows'])}", flush=True)
            continue
        vp, rm, va, best_sd = train_fold(seed, k)
        oof[va] = vp; filled[va] = True
        torch.save(best_sd, mpath)                            # <-- trained model weights
        np.savez_compressed(cache, rows=va.astype(np.int64), pred=vp.astype(np.float64))
        _folds_done += 1
        _el = time.time() - _run_t0
        _eta = _el / max(_folds_done, 1) * (_folds_total - _folds_done)
        print(f"seed{seed} fold{k} [fit] val={len(va)} rmse={rm:.4f} saved {mpath.name}  "
              f"[{_folds_done}/{_folds_total} fold  elapsed {_el/60:.0f}min  ETA~{_eta/60:.0f}min]", flush=True)
    seed_oofs[seed] = oof
    rm = float(np.sqrt(np.mean((oof[filled] - y[filled]) ** 2)))
    print(f"== seed{seed} OOF pooled RMSE(target) = {rm:.4f} "
          f"(filled {int(filled.sum())}/{N}) ({time.time()-tt:.0f}s)", flush=True)


In [ ]:
# =====================================================================
# COMPLEMENTARITY ANALYSIS  (self-contained on Kaggle; shows the verdict after 1 fold)
# Compares this run's tabnn OOF against the OTHER legs' OOFs shipped in artifacts_v93:
#   gbm_l1_oof.npy (GBM L1) | nn_oof_combo.npz (mdn2_s* mean = TCN) | l5_oof.npz (l5_mean = L5)
# Restricted to the val rows of the fold(s) actually trained. nnls blend on those rows.
# =====================================================================
from scipy.optimize import nnls

# rows tabnn actually predicted this run (union of trained folds' val indices)
val_rows = np.unique(np.concatenate([folds[k][1] for k in FOLDS_TO_RUN]))
tabv = np.mean([seed_oofs[s][val_rows] for s in SEEDS], axis=0).astype(np.float64)
yv   = y[val_rows].astype(np.float64)
print(f"complementarity on {len(val_rows)} val rows (folds {FOLDS_TO_RUN}, seeds {SEEDS})")

# ---- load other legs' OOFs from the uploaded artifacts_v93 (align by row index) ----
legs = {}
try:
    legs["gbm"] = np.load(ART / "gbm_l1_oof.npy", mmap_mode="r")[val_rows].astype(np.float64)
except Exception as e:
    print("  [skip gbm]", e)
try:
    _z = np.load(ART / "nn_oof_combo.npz")
    _ks = [k for k in _z.files if k.startswith("mdn2_s")]
    legs["tcn"] = np.mean([_z[k][val_rows] for k in _ks], axis=0).astype(np.float64)
except Exception as e:
    print("  [skip tcn]", e)
try:
    legs["l5"] = np.load(ART / "l5_oof.npz")["l5_mean"][val_rows].astype(np.float64)
except Exception as e:
    print("  [skip l5]", e)

def _rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))

# ---- (a) correlation + (b) single-leg RMSE ----
print("---- single-leg RMSE(target) & corr(tabnn, leg) [lower corr = more complementary] ----")
print(f"  tabnn : rmse={_rmse(tabv, yv):.4f}")
for name, v in legs.items():
    corr = float(np.corrcoef(tabv, v)[0, 1])
    print(f"  {name:5s}: rmse={_rmse(v, yv):.4f}   corr(tabnn,{name})={corr:+.3f}")

# ---- (c) nnls blend: 3-way {gbm,tcn,l5} vs 4-way {+tabnn} ----
base_names = [n for n in ("gbm", "tcn", "l5") if n in legs]
if base_names:
    A3 = np.column_stack([legs[n] for n in base_names])
    w3, _ = nnls(A3, yv); rmse3 = _rmse(A3 @ w3, yv)
    A4 = np.column_stack([legs[n] for n in base_names] + [tabv])
    w4, _ = nnls(A4, yv); rmse4 = _rmse(A4 @ w4, yv)
    tab_w = float(w4[-1]); improve = rmse3 - rmse4
    print("---- nnls blend (val rows only) ----")
    print(f"  3-way {base_names}         : rmse={rmse3:.4f}  w={[round(float(x),3) for x in w3]}")
    print(f"  4-way {base_names + ['tabnn']} : rmse={rmse4:.4f}  w={[round(float(x),3) for x in w4]}")
    print(f"  tabnn blend weight = {tab_w:.4f} | RMSE improvement 3->4-way = {improve:+.4f}")
    # ---- (d) verdict ----
    min_corr = min((float(np.corrcoef(tabv, legs[n])[0, 1]) for n in base_names), default=1.0)
    is_comp = (tab_w > 1e-4) and (improve > 1e-4)
    print("=====================================================================")
    if is_comp:
        print(f"VERDICT: COMPLEMENTARY = YES  (tabnn nonzero weight {tab_w:.3f}, "
              f"blend RMSE -{improve:.4f}, min corr {min_corr:+.3f})")
    else:
        print(f"VERDICT: COMPLEMENTARY = NO  (tabnn weight {tab_w:.3f}, "
              f"blend delta {improve:+.4f}, min corr {min_corr:+.3f})")
    print("  (1-fold read; confirm with full SEEDS[0,1,2] x FOLDS range(5) before trusting.)")
    print("=====================================================================")
else:
    print("no base legs found in artifacts_v93 -> cannot run nnls blend verdict.")


In [ ]:
# =====================================================================
# Save OOF (tabnn_oof.npz) + reconstruction config (tabnn_config.json) + final summary
# =====================================================================
save = {f"tab_s{s}": seed_oofs[s].astype(np.float32) for s in SEEDS}
mean_oof = np.mean([seed_oofs[s] for s in SEEDS], axis=0)
save["tab_mean"] = mean_oof.astype(np.float32)     # seed-ensemble OOF (per-row, TARGET units)
save["y"] = y.astype(np.float32)
# rows actually predicted this run (zeros elsewhere when running a subset of folds)
_filled = np.zeros(N, bool)
_filled[np.unique(np.concatenate([folds[k][1] for k in FOLDS_TO_RUN]))] = True
save["filled"] = _filled
oof_fp = WORK / "tabnn_oof.npz"
np.savez_compressed(oof_fp, **save)
print(f"OOF filled rows = {int(_filled.sum())}/{N} (folds {FOLDS_TO_RUN}); use the 'filled' mask when stacking.")

# everything needed to rebuild the exact preprocessing + model for later inference/submit
cfg = dict(
    feats_nn=feats_nn,                             # TOPK-selected columns, in order
    n_feat=F_IN,
    DIM=DIM, N_BLOCK=N_BLOCK, N_HEAD=N_HEAD, FF_MULT=FF_MULT, DROP=DROP, TOPK=TOPK,
    med=[float(v) for v in med],                   # robust-scale center
    scale=[float(v) for v in scl],                 # robust-scale (IQR/1.349)
    t_mean=t_mean, t_scale=t_scale,                # target standardization
    AMP=AMP, BATCH=BATCH, seeds=SEEDS,
    fold="GroupKFold(5).split(np.zeros(N), y, groups=well)  # identical to L1 GBM / L2 TCN / L5",
    preprocess="X = clip((X - med)/scale, -8, 8); nan->0, +inf->8, -inf->-8",
    decode="pred_target = pred_std * t_scale + t_mean",
    model_files="tabnn_models/tab_s{seed}_f{fold}.pt  (EMA state_dict; load into FTTransformer(n_feat,DIM,N_BLOCK,N_HEAD,FF_MULT,DROP))",
)
cfg_fp = WORK / "tabnn_config.json"
Path(cfg_fp).write_text(json.dumps(cfg))

print("saved:", oof_fp)
print("saved:", cfg_fp)
print("---- OOF pooled RMSE (target units, filled rows only) ----")
for s in SEEDS:
    print(f"  seed{s}: {np.sqrt(np.mean((seed_oofs[s][_filled]-y[_filled])**2)):.4f}")
rm_ens = float(np.sqrt(np.mean((mean_oof[_filled] - y[_filled]) ** 2)))
print(f"== {len(SEEDS)}-seed ensemble OOF pooled RMSE(target, filled) = {rm_ens:.4f}")
print("---- saved model files ----")
_mfiles = sorted(MODELS_DIR.glob("*.pt"))
for p in _mfiles:
    print(f"  {p}  ({p.stat().st_size/1e6:.1f} MB)")
print(f"({len(_mfiles)} model files, expected {len(SEEDS)*N_SPLITS})")
print("done.")


## Download the outputs

Everything is written under `/kaggle/working` (visible in the notebook's **Output** tab):

| file | contents |
|------|----------|
| `tabnn_oof.npz` | keys `tab_s0`, `tab_s1`, `tab_s2`, `tab_mean`, `y` — per-row OOF predictions in **TARGET units** (`tab_mean` = 3-seed ensemble). Stacks with the L1 GBM / L2 TCN / L5 OOFs (same `GroupKFold(5)` rows). |
| `tabnn_models/tab_s{seed}_f{fold}.pt` | trained **EMA weights** (state_dict) for each of the 15 (seed x fold) models. |
| `tabnn_config.json` | `feats_nn` (selected columns in order), model dims, robust-scale `med`/`scale`, `t_mean`/`t_scale`, AMP/BATCH/seeds, fold + preprocessing notes — enough to rebuild the exact preprocessing + model for inference. |
| `tabnn_valpred/tab_s{seed}_f{fold}_valpred.npz` | per-fold val-pred cache (used for **resume**). |

**Rebuild for inference:** load `tabnn_config.json`, build `FTTransformer(n_feat, DIM, N_BLOCK, N_HEAD, FF_MULT, DROP)`,
`load_state_dict(torch.load('tab_s{seed}_f{fold}.pt'))`, scale test features with `med`/`scale`
(clip to +-8, nan->0), forward, then decode `pred_target = pred_std * t_scale + t_mean`.
Ensemble the folds/seeds (mean).

To rerun from scratch, clear the `tabnn_models/` and `tabnn_valpred/` dirs (otherwise completed folds are skipped).